# Prompt engineering · instruction-only baselines

| | |
|---|---|
| **Project** | EdgeVLM-Drive — resource-aware vision-language driver monitoring on NVIDIA Jetson Nano |
| **Model** | SmolVLM-256M (single-crop, already adapted) |
| **Purpose** | Evaluates instruction-only prompts through the LangGraph pipeline before any further fine-tuning (research question RQ1). |
| **Input** | driver-state test frames and the prompt variants defined in the notebook |
| **Output** | per-prompt predictions and macro-F1 / unsafe-frame miss rate |
| **Type** | Experiment; cell outputs from the original run are retained |


# EdgeVLM-Drive Prompting with LangGraph

This notebook evaluates four prompt types on the fine-tuned SmolVLM-256M model.
The existing `langgraph_pipeline.py` is imported from `/home/jovyan/EdgeVLM_finetune`.
Ground-truth labels are used only after inference for scoring; they are never sent to the model.


## Full-test prompting evaluation
This copy evaluates all 3,579 test images for every prompt. It does not retrain the model.
Results are saved under `/home/jovyan/EdgeVLM_prompting_full_test/` so the earlier 100-image results remain unchanged.

In [1]:
from pathlib import Path
import sys
import time

import cv2
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path('/home/jovyan/EdgeVLM_finetune')
PROMPT_ROOT = Path('/home/jovyan/EdgeVLM_prompting_full_test')
PROMPT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))

from langgraph_pipeline import VLMRuntime, build_graph, new_frame_state

CLASSES = ['NORMAL', 'DROWSY', 'LOOKING_AWAY', 'PHONE_USAGE']
TEST_CSV = Path('/home/jovyan/splits/test.csv')
test_df = pd.read_csv(TEST_CSV)

def resolve_frame_path(frame_path):
    path = Path(str(frame_path).replace('\\', '/'))
    candidates = [
        path,
        Path('/home/jovyan') / path,
        Path('/home/jovyan/dmd_data_downloads') / path,
        Path('/home/jovyan/splits_augmented_v2') / path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(str(frame_path))

# Evaluate the complete test split for the final comparison.
USE_BALANCED_SUBSET = False
if USE_BALANCED_SUBSET:
    eval_df = pd.concat(
        [group.head(25) for _, group in test_df.groupby('label')]
    ).reset_index(drop=True)
else:
    eval_df = test_df.copy()

print('Evaluation images:', len(eval_df))
print(eval_df['label'].value_counts())


Evaluation images: 3579
label
NORMAL          2151
PHONE_USAGE      958
LOOKING_AWAY     302
DROWSY           168
Name: count, dtype: int64


/opt/conda/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoModelForImageTextToText
from peft import PeftModel

BASE_MODEL_ID = 'HuggingFaceTB/SmolVLM-256M-Instruct'

# Use the unweighted adapter first. To test the class-weighted model, replace this path.
ADAPTER_DIR = PROJECT_ROOT / 'smolvlm_256m_lora_augmented_deduplicated'

device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device_type == 'cuda' else torch.float32

processor = AutoProcessor.from_pretrained(str(ADAPTER_DIR))
model_kwargs = {'torch_dtype': dtype, 'low_cpu_mem_usage': True}
if device_type == 'cuda':
    model_kwargs['device_map'] = 'auto'

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    **model_kwargs,
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
MODEL_DEVICE = next(model.parameters()).device

print('Model:', BASE_MODEL_ID)
print('Adapter:', ADAPTER_DIR)
print('Device:', MODEL_DEVICE)


2026-09-10 20:21:32.065866: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/conda/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


Model: HuggingFaceTB/SmolVLM-256M-Instruct
Adapter: /home/jovyan/EdgeVLM_finetune/smolvlm_256m_lora_augmented_deduplicated
Device: cuda:0


In [3]:
PROMPTS = {
    'minimal_zero_shot': '''
Classify the driver's visible state as exactly one label:
NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE.

Return only one label.
''',
    'definition_zero_shot': '''
Classify the driver's visible state using exactly one label.

NORMAL: the driver appears attentive and faces the road.
DROWSY: the driver shows closed or drooping eyes, yawning, or sleepiness.
LOOKING_AWAY: the driver's face or gaze is directed away from the road.
PHONE_USAGE: the driver is holding or using a phone.

Allowed labels: NORMAL, DROWSY, LOOKING_AWAY, PHONE_USAGE.
Return only one label and no explanation.
''',
    'contrastive': '''
Classify the driver's state using exactly one label.

Use PHONE_USAGE when a phone is visible.
Use DROWSY when the eyes appear closed or drooping, or the driver is yawning.
Use LOOKING_AWAY when the driver is alert but looking away from the road.
Use NORMAL only when none of these unsafe behaviors is visible.

Allowed labels: NORMAL, DROWSY, LOOKING_AWAY, PHONE_USAGE.
Return exactly one label and nothing else.
''',
    'cue_guided': '''
Analyze the driver image in this order:
1. Check whether a phone is visible or being used.
2. Check for closed eyes, drooping eyelids, yawning, or sleepiness.
3. Check whether the driver is looking away from the road.
4. If none of these cues is present, select NORMAL.

Return exactly one label: NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE.
Do not return an explanation or punctuation.
''',
}


def crop_to_pil(crop):
    from PIL import Image
    if crop is None:
        return None
    array = np.asarray(crop)
    if array.dtype != np.uint8:
        array = np.clip(array, 0, 255).astype(np.uint8)
    return Image.fromarray(array).convert('RGB')


class NotebookSmolVLMRuntime(VLMRuntime):
    def __init__(self, model, processor, prompt):
        self.model = model
        self.processor = processor
        self.prompt = prompt
        self.calls = 0
        self.call_latencies_ms = []
        self.outputs = []

    def reset(self):
        self.calls = 0
        self.call_latencies_ms = []
        self.outputs = []

    def generate_label(self, face_crop, full_crop):
        # Single face crop matches the format used during SmolVLM fine-tuning.
        image = crop_to_pil(face_crop if face_crop is not None else full_crop)
        if image is None:
            raise RuntimeError('No valid crop was produced')

        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': self.prompt},
            ],
        }]

        chat_prompt = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
        )

        try:
            inputs = self.processor(
                text=chat_prompt,
                images=[image],
                return_tensors='pt',
                size={'longest_edge': 512},
                max_image_size={'longest_edge': 512},
            )
        except TypeError:
            inputs = self.processor(
                text=chat_prompt,
                images=[image],
                return_tensors='pt',
            )

        inputs = {
            key: value.to(MODEL_DEVICE) if torch.is_tensor(value) else value
            for key, value in inputs.items()
        }

        start = time.perf_counter()
        with torch.inference_mode():
            generated = self.model.generate(
                **inputs,
                max_new_tokens=12,
                do_sample=False,
            )
        self.call_latencies_ms.append((time.perf_counter() - start) * 1000)
        self.calls += 1

        input_length = inputs['input_ids'].shape[-1]
        output = self.processor.batch_decode(
            generated[:, input_length:],
            skip_special_tokens=True,
        )[0].strip()
        self.outputs.append(output)
        return output


In [4]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix


def evaluate_prompt(prompt_name, prompt_text, dataframe):
    runtime = NotebookSmolVLMRuntime(model, processor, prompt_text)
    graph = build_graph(runtime)
    records = []

    for index, row in enumerate(dataframe.itertuples(index=False), start=1):
        image_path = resolve_frame_path(row.frame_path)
        frame_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if frame_bgr is None:
            raise RuntimeError('Could not read image: ' + str(image_path))

        runtime.reset()
        start = time.perf_counter()

        # landmarks=None forces the VLM path. The true label is not included.
        state = new_frame_state(
            frame_id=index,
            raw_frame=frame_bgr,
            landmarks=None,
            scheduled_interval_s=10.0,
        )
        result = graph.invoke(state)

        final_raw = (result.get('vlm_raw_output') or '').strip()
        errors = result.get('errors', [])
        records.append({
            'frame_path': str(row.frame_path),
            'true_label': str(row.label),
            'raw_output': final_raw,
            'parsed_label': result.get('label'),
            'exact_label_valid': final_raw.upper() in CLASSES,
            'parser_failure': any('exhausted retries' in str(e) for e in errors),
            'retry_used': runtime.calls > 1,
            'vlm_calls': runtime.calls,
            'vlm_latency_ms': sum(runtime.call_latencies_ms),
            'graph_latency_ms': (time.perf_counter() - start) * 1000,
            'gate_reason': result.get('gate_reason'),
            'errors': ' | '.join(map(str, errors)),
        })

        if index % 25 == 0:
            print(f'{prompt_name}: {index}/{len(dataframe)}')

    predictions = pd.DataFrame(records)
    y_true = predictions['true_label'].tolist()
    y_pred = predictions['parsed_label'].tolist()
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)

    unsafe = predictions['true_label'] != 'NORMAL'
    normal = predictions['true_label'] == 'NORMAL'
    unsafe_miss = (predictions.loc[unsafe, 'parsed_label'] != predictions.loc[unsafe, 'true_label']).mean() * 100
    false_alarm = (predictions.loc[normal, 'parsed_label'] != 'NORMAL').mean() * 100

    metrics = {
        'prompt': prompt_name,
        'images': len(predictions),
        'exact_label_validity_rate': predictions['exact_label_valid'].mean() * 100,
        'parser_failure_rate': predictions['parser_failure'].mean() * 100,
        'retry_rate': predictions['retry_used'].mean() * 100,
        'overall_accuracy': accuracy_score(y_true, y_pred) * 100,
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred) * 100,
        'macro_f1': f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0) * 100,
        'unsafe_state_miss_rate': unsafe_miss,
        'false_alarm_rate': false_alarm,
        'mean_graph_latency_ms': predictions['graph_latency_ms'].mean(),
        'p95_graph_latency_ms': predictions['graph_latency_ms'].quantile(0.95),
        'mean_vlm_latency_ms': predictions['vlm_latency_ms'].mean(),
        'mean_vlm_calls_per_image': predictions['vlm_calls'].mean(),
    }

    print('\n', prompt_name)
    print(pd.Series(metrics))
    print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES))
    return predictions, metrics


In [ ]:
all_metrics = []
all_predictions = {}

for prompt_name, prompt_text in PROMPTS.items():
    predictions, metrics = evaluate_prompt(prompt_name, prompt_text, eval_df)
    all_metrics.append(metrics)
    all_predictions[prompt_name] = predictions
    predictions.to_csv(
        PROMPT_ROOT / f'{prompt_name}_predictions.csv',
        index=False,
    )

metrics_df = pd.DataFrame(all_metrics).sort_values('macro_f1', ascending=False)
metrics_df.to_csv(PROMPT_ROOT / 'prompt_comparison_metrics.csv', index=False)
display(metrics_df)
print('Results saved in:', PROMPT_ROOT)


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


minimal_zero_shot: 25/3579
minimal_zero_shot: 50/3579
minimal_zero_shot: 75/3579
minimal_zero_shot: 100/3579
minimal_zero_shot: 125/3579
minimal_zero_shot: 150/3579
minimal_zero_shot: 175/3579
minimal_zero_shot: 200/3579
minimal_zero_shot: 225/3579
minimal_zero_shot: 250/3579
minimal_zero_shot: 275/3579
minimal_zero_shot: 300/3579
minimal_zero_shot: 325/3579
minimal_zero_shot: 350/3579
minimal_zero_shot: 400/3579
minimal_zero_shot: 425/3579
minimal_zero_shot: 450/3579
minimal_zero_shot: 475/3579
minimal_zero_shot: 500/3579
minimal_zero_shot: 525/3579
minimal_zero_shot: 550/3579
minimal_zero_shot: 575/3579
minimal_zero_shot: 600/3579
minimal_zero_shot: 625/3579
minimal_zero_shot: 650/3579
minimal_zero_shot: 675/3579
minimal_zero_shot: 700/3579
minimal_zero_shot: 725/3579
minimal_zero_shot: 750/3579
minimal_zero_shot: 775/3579
minimal_zero_shot: 800/3579
minimal_zero_shot: 825/3579
minimal_zero_shot: 850/3579
minimal_zero_shot: 875/3579
minimal_zero_shot: 900/3579
minimal_zero_shot: 925/